# Adapter + retrieval: does giving the adapter what I said in similar conversations make it answer better?

Baselines.ipynb: the adapter sounds like me (style position 0.96 / 0.81) but barely answers the conversation (relevance 0.27), and copying my real reply from the most similar validation conversation answers a bit better (0.37), although the paired difference isn't demonstrable. The literature calls this split "the adapter gives the form, retrieval the content". This notebook tests the combination:

- The same 0.5B adapter (run1_5ep, step 2,500) and the same 1,133 test contexts, temperature 1.0.
- In the system prompt, my replies in the 3 most similar conversations of the **validation** pool (the TF-IDF lookup of `scripts/baseline_retrieval.py`). Validation, not train, so the style classifier (trained on train replies) can't recognise copied text. `scripts/generate.py --retrieve 3`.
- Style with the same frozen classifiers, relevance with the same Qwen2.5-7B judge and cache as Baselines.ipynb, and paired differences against the adapter alone and retrieval alone.

The adapter was never trained with extra text in the system prompt, so this is retrieval at inference time only (no retraining).

Where it runs: generation and relevance on the NAS (RTX 3080; evaluation there matches the Mac to 3e-4 nats, calib-cuda3), style on the Mac (the mDeBERTa classifier lives there; re-running the notebook skips what's done).

## Prediction (written before running)

Fede: the adapter won't ignore the extra context: even if it wasn't trained with it, it's information it has to work with. It should work somewhat better: relevance position around 0.45-0.50 (adapter alone 0.27, retrieval alone 0.37). And the replies must not be exact copies of the retrieved ones.

How it's checked:
- Relevance position of adapter + retrieval, and paired differences in context gain against the adapter alone and retrieval alone. "Better than the adapter" = that paired interval excludes 0. Caveat written beforehand: retrieval vs adapter (+0.044 nats) wasn't demonstrable with these intervals (about ±0.1 nats), so 0.45-0.50 against retrieval's 0.37 may be right and still not demonstrable.
- Copying: share of generated replies identical to one of the 3 retrieved, and share nearly identical (character similarity >= 0.9). Counts only, no text.
- Style must stay where the adapter alone is (0.96 / 0.81).

In [1]:
# Heavy steps run scripts through run() and resume: re-running skips what is done. Run headless with:
#   cd experiments && nohup uv run jupyter nbconvert --to notebook --execute --inplace \
#       --ExecutePreprocessor.timeout=-1 AdapterRetrieval.ipynb > AdapterRetrieval.log 2>&1 &
import json, os, subprocess
from collections import defaultdict
from pathlib import Path
import numpy as np
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")             # NAS: TLM_PY=/opt/venv/bin/python
JUDGE = "mlx-community/Qwen2.5-7B-Instruct-4bit"
K = 3
NAME = f"adapter_rag{K}"

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

## 1. Generate: adapter + my replies in the 3 most similar validation conversations

In [2]:
run(f"{PY} scripts/generate.py --name {NAME} --adapter adapters/run1_5ep --step 2500 --retrieve {K}")

$ uv run python scripts/generate.py --name adapter_rag3 --adapter adapters/run1_5ep --step 2500 --retrieve 3


1133 replies in data/style/generations_adapter_rag3.jsonl


## 2. Relevance, judged by Qwen2.5-7B (same cache as Baselines.ipynb: only the new set is scored)

In [3]:
if not (R / "relevance_rag.json").exists():
    run(f"{PY} scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval_valid {NAME} "
        f"--scorer {JUDGE} --cache experiments/results/relevance_nll_7b.json --out experiments/results/relevance_rag.json")

## 3. Style: the same frozen classifiers (Mac)

In [4]:
# On the NAS (TLM_PY set) this is skipped: the mDeBERTa classifier is on the Mac and runs on MPS.
if not (R / "style_rag.json").exists() and "TLM_PY" not in os.environ:
    run(f"{PY} scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval_valid {NAME} --transformer --out experiments/results/style_rag.json")

$ uv run python scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval_valid adapter_rag3 --transformer --out experiments/results/style_rag.json


.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.


  warnings.warn(


length (test AUC on real messages 0.551)


  base       classified as me   7.8% [5.6%, 9.5%]


  adapter    classified as me  41.9% [39.6%, 44.4%]


... (23 outputs left out)

  friends    classified as me  26.5% [19.1%, 34.2%]


  position of adapter: 0.81 [0.69, 0.90]


  position of retrieval_valid: 1.09 [0.98, 1.19]


  position of adapter_rag3: 0.85 [0.75, 0.95]


## Summary

In [5]:
import sys
sys.path.insert(0, str(ROOT))
from talk_like_me.config import DATA
from talk_like_me.io import read_jsonl
from talk_like_me.stats import interval, resample

names = {"adapter": "adapter alone", "adapter t1.0": "adapter alone", "retrieval_valid": "retrieval alone (valid pool)", NAME: f"adapter + retrieval (K={K})"}
rel = json.load(open(R / "relevance_rag.json"))
style = json.load(open(R / "style_rag.json"))["scores"] if (R / "style_rag.json").exists() else None
print(f"{'set':<30} {'style TF-IDF':>14} {'style mDeBERTa':>15} {'relevance (7B)':>16}")
for k in ["adapter", "retrieval_valid", NAME]:
    r = rel["positions"]["adapter t1.0" if k == "adapter" else k]
    cols = [f"{s[f'position {k}']['value']:>6.2f} [{s[f'position {k}']['ci'][0]:.2f},{s[f'position {k}']['ci'][1]:.2f}]"
            for s in (style["tfidf"], style["transformer"])] if style else ["(Mac)", "(Mac)"]
    print(f"{names[k]:<30} {cols[0]:>14} {cols[1]:>15} {r['value']:>6.2f} [{r['ci'][0]:.2f},{r['ci'][1]:.2f}]")
print("style position: 0 = base model, 1 = me | relevance position: 0 = my reply to another conversation, 1 = my real reply")

# Paired differences in relevance gain: same contexts, same bootstrap samples of sessions.
nll = json.load(open(R / "relevance_nll_7b.json"))
sessions = [r["session"] for r in read_jsonl(DATA / "style" / "test.jsonl") if r["label"] == 1]
sums = {}
for name, rows in nll.items():
    s_ = defaultdict(lambda: np.zeros(3))
    for (a, n, b, _), s in zip(rows, sessions):
        s_[s] += (a, b, n)
    sums[name] = dict(s_)
gain = lambda name, chosen: (lambda t: (t[1] - t[0]) / t[2])(sum(sums[name][s] for s in chosen))
samples, all_s = resample(set(sessions)), sorted(set(sessions))
print()
for a, b in [(NAME, "adapter t1.0"), (NAME, "retrieval_valid")]:
    boot = [gain(a, c) - gain(b, c) for c in samples]
    lo, hi = interval(boot)
    print(f"relevance gain, {names[a]} - {names[b]}: {gain(a, all_s) - gain(b, all_s):+.3f} nats [{lo:+.3f}, {hi:+.3f}]")

set                              style TF-IDF  style mDeBERTa   relevance (7B)
adapter alone                    0.96 [0.84,1.08]   0.81 [0.69,0.90]   0.27 [0.09,0.42]
retrieval alone (valid pool)     1.05 [0.95,1.14]   1.09 [0.98,1.19]   0.37 [0.14,0.64]
adapter + retrieval (K=3)        1.00 [0.83,1.18]   0.85 [0.75,0.95]   0.34 [0.17,0.46]
style position: 0 = base model, 1 = me | relevance position: 0 = my reply to another conversation, 1 = my real reply

relevance gain, adapter + retrieval (K=3) - adapter alone: +0.027 nats [-0.040, +0.109]
relevance gain, adapter + retrieval (K=3) - retrieval alone (valid pool): -0.017 nats [-0.134, +0.077]


## Copying: how many replies are (nearly) the retrieved ones

In [6]:
import difflib
sys.path.insert(0, str(ROOT / "scripts"))
from baseline_retrieval import similar
from talk_like_me.config import DATASET

contexts = read_jsonl(DATASET / "session_ctx20" / "test.jsonl")
retrieved = similar("valid", contexts, K)
gens = [g["text"] for g in read_jsonl(DATA / "style" / f"generations_{NAME}.jsonl")]
norm = lambda t: " ".join(t.lower().split())
exact = sum(any(norm(g) == norm(r) for r in rs) for g, rs in zip(gens, retrieved))
near = sum(max(difflib.SequenceMatcher(None, norm(g), norm(r)).ratio() for r in rs) >= 0.9 for g, rs in zip(gens, retrieved))
print(f"identical to a retrieved reply: {exact}/{len(gens)} ({exact / len(gens):.1%})")
print(f"nearly identical (similarity >= 0.9): {near}/{len(gens)} ({near / len(gens):.1%})")

identical to a retrieved reply: 0/1133 (0.0%)
nearly identical (similarity >= 0.9): 0/1133 (0.0%)


## Conclusion

**Prediction (Fede):** the adapter uses the extra context and answers somewhat better, relevance position around 0.45-0.50, without copying the retrieved replies.

**Half right: it doesn't copy and keeps my style, but it doesn't answer better.**
- **Relevance 0.34 [0.17, 0.46]**, between the adapter alone (0.27) and retrieval alone (0.37), well short of 0.45-0.50. Paired against the adapter alone: +0.027 nats [-0.040, +0.109], so the criterion written beforehand (interval excluding 0) is not met. Against retrieval alone: -0.017 [-0.134, +0.077].
- **No copying:** 0 of 1,133 replies identical to one of the 3 retrieved, 0 nearly identical (character similarity >= 0.9). It writes its own replies.
- **Style unchanged:** 1.00 / 0.85 against 0.96 / 0.81 alone; the retrieved replies don't pull it away from sounding like me (nor closer: the intervals overlap).

**Reading:** the adapter doesn't ignore the extra text (relevance moves a little in the predicted direction and style holds), but 3 replies of mine from *other* conversations don't tell it much about what to say in *this* one. Retrieval alone works because it hands over a whole real reply; mixed into the prompt as examples, they act more like a few-shot reminder of how I write, which the adapter already knows.

**Limits and what's left untried.**
- The adapter was never trained with retrieved replies in its prompt. Training with them (so it learns to use them) would be a different experiment; it was considered and closed (7 Nov 2026): not relevant to the project's question.
- Only my replies were retrieved, without the conversations they answered, and only K=3.
- The judge measures coherence with the conversation, not whether the content is true (Baselines.ipynb): retrieval could help with facts in a way this metric can't see.
